# Exploration du parc automobile (SDES)

Source: SDES, parc de véhicules routiers au niveau communal (data.gouv.fr, Licence Ouverte). 
Ce qui nous intéresse ici est de récupérer les données liées aux voitures particulières (VP) des particuliers, au 1er janvier 2017 et au 1er janvier 2023. Les voitures de société et de location sont exclues (elles sont immatriculées à l'adresse de l'entreprise ce qui sous-estimerait ou sur-estimerait le parc automobile de certaines communes).

In [3]:
import os
import sys

import yaml

# on se place à la racine du projet
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

# paramètres communs du projet
with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

from src.collecte import get_parc_auto

parc = get_parc_auto(config["urls"]["parc_auto"], config["annee_debut"], config["annee_fin"])

print(parc.shape)
print(parc.dtypes)
parc.head()

(523325, 6)
COMMUNE_CODE      str
COMMUNE_NOM       str
CARBURANT         str
CRIT_AIR          str
PARC_2017       int64
PARC_2023       int64
dtype: object


,COMMUNE_CODE,COMMUNE_NOM,CARBURANT,CRIT_AIR,PARC_2017,PARC_2023
0,00000,Inconnu,Essence HNR,Crit'Air 1,1,143
1,00000,Inconnu,Gaz,Crit'Air 1,3,3
2,00000,Inconnu,Diesel,Crit'Air 2,307,518
3,00000,Inconnu,Diesel,Crit'Air 3,981,78
4,00000,Inconnu,Diesel,Crit'Air 4,44,28


In [ ]:
# 1.on veut compter le nombre de commune pour pouvoir les regrouper en une seule ligne (une ligne par commune)
print("communes différentes :", parc["COMMUNE_CODE"].nunique())

# 2. on compte les voitures sans commune qui vont être retirées
inconnu = parc["COMMUNE_CODE"].str.endswith("000")
for annee in ["PARC_2017", "PARC_2023"]:
    part = parc.loc[inconnu, annee].sum() / parc[annee].sum() * 100
    print(f"{annee} : {part:.2f} % des voitures ont un code inconnu")

# 3. voir si on peut trouver les arrondissements parisiens, marsaillais et lyonnais à partir des codes INSEE
arrondissements = parc["COMMUNE_CODE"].str.match(r"^(751|6938|132)")
print("codes d'arrondissements :", sorted(parc.loc[arrondissements, "COMMUNE_CODE"].unique())[:5], "...")

# 4. valeurs manquantes et total national
print("manquants :", parc.isna().sum().to_dict())
print("voitures 2017 :", parc["PARC_2017"].sum(), "| 2023 :", parc["PARC_2023"].sum())

communes différentes : 35017
PARC_2017 : 0.02 % des voitures ont un code inconnu
PARC_2023 : 0.05 % des voitures ont un code inconnu
codes d'arrondissements : ['13201', '13202', '13203', '13204', '13205'] ...
manquants : {'COMMUNE_CODE': 0, 'COMMUNE_NOM': 48, 'CARBURANT': 0, 'CRIT_AIR': 0, 'PARC_2017': 0, 'PARC_2023': 0}
voitures 2017 : 35122936 | 2023 : 35859860


## Constats

Pour les communes, on a 523 000 lignes pour 35 017 codes. Ainsi, pour avoir le nombre de voitures d'une commune, il faut additionner toutes ses lignes.
Les voitures au code de commune inconnu correspond à 0.05% du total au maximum. C'est négligeable. On peut donc se permettre de les retirer, lors du nettoyage.
En terme de parc automobile: 35,1 millions de voitures en 2017, 35,9 millions en 2023 (+2 %).
  L'ordre de grandeur est cohérent pour les voitures des particuliers.
- **À vérifier** : 35 017 codes contre 34 746 communes en métropole. L'écart vient des
  arrondissements, des DOM, des codes inconnus, et peut-être d'anciennes communes fusionnées
  (le SDES ne précise pas le millésime de ses codes communes).

## Ce que fera le nettoyage (`clean_parc_auto`)

1. retirer les codes inconnus ;
2. ramener les arrondissements à leur ville ;
3. additionner, par commune, toutes les lignes pour 2017 et pour 2023 ;
4. garder la métropole ;
5. traiter les anciens codes de communes, s'il y en a (voir la vérification suivante) ;
6. vérifier par des `assert` : code commune unique, sur 5 caractères.

In [ ]:
import pandas as pd

# on modifie notre base de données pour que Paris, Marseille et Lyon devienne des communes à part entière. 
codes = parc["COMMUNE_CODE"].copy()
codes[codes.str.startswith("751")] = "75056"   #    Paris
codes[codes.str.startswith("6938")] = "69123"   # Lyon
codes[codes.str.startswith("132")] = "13055"   #   Marseille

# voitures 2023 par commune, sans les codes inconnus
voitures = parc[~inconnu].assign(code=codes).groupby("code")["PARC_2023"].sum()

# communes des contours (métropole, 2026), avec leur nom
contours = pd.read_parquet("data/clean/contours_communes.parquet",
                           columns=["code_commune", "nom_commune"])

# 1. codes du parc absents des contours
absents = voitures[~voitures.index.isin(contours["code_commune"])]
dom = absents.index.str.startswith("97")
print("codes du parc absents des contours :", len(absents), "dont DOM :", dom.sum())
print(f"part des voitures 2023 hors DOM et absentes des contours : {absents[~dom].sum() / voitures.sum() * 100:.2f} %")
print("exemples hors DOM :", absents[~dom].index[:15].tolist())

# 2. communes des contours absentes du parc
sans_voiture = contours[~contours["code_commune"].isin(voitures.index)]
print("communes des contours absentes du parc :", len(sans_voiture))
sans_voiture.head(20)

codes du parc absents des contours : 134 dont DOM : 134
part des voitures 2023 hors DOM et absentes des contours : 0.00 %
exemples hors DOM : []
communes des contours absentes du parc : 4


,code_commune,nom_commune
16028,55050,Bezonvaux
16043,55139,Cumières-le-Mort-Homme
16062,55239,Haumont-près-Samogneux
16071,55307,Louvemont-Côte-du-Poivre


- **Correspondance avec les contours** : hors DOM, tous les codes du parc existent dans les
  contours 2026 (0,00 % des voitures perdues).
- **4 communes des contours n'ont aucune voiture dans le parc** (Bezonvaux, Cumières-le-Mort-Homme,
  Haumont-près-Samogneux, Louvemont-Côte-du-Poivre, dans la Meuse). On va donc les retirer de la base, car elles sont négligeables par rapport au reste de la base. 

In [ ]:
from src.nettoyage import clean_parc_auto

voitures = clean_parc_auto(parc, 2017, 2023)
print(voitures.shape)
print(voitures[["voitures_2017", "voitures_2023"]].sum())
voitures[voitures["code_commune"].isin(["75056", "69123", "13055"])]

(34742, 3)
voitures_2017    34350825
voitures_2023    35035890
dtype: int64


,code_commune,voitures_2017,voitures_2023
4338,13055,338308,324370
27035,69123,173356,159615
29190,75056,462773,398215


## Bilan

Fichier propre : `data/clean/parc_auto.parquet`
On remarque q'il y a en premier constat une baisse du nombre de voitures à Paris, Lyon et Marseille. 

In [ ]:
voitures.to_parquet("data/clean/parc_auto.parquet")
print("enregistré dans :", os.getcwd())

enregistré dans : /home/onyxia/work/le-rail-suit-il-la-France-qui-grandit-
